#### Teoría de los Circuitos 2
# Trabajo semanal 6
#### Leandro Robledo

A continuación se muestran 2 circuitos conocidos en un esquema de 3 nodos externos.  Tener en cuenta que para el análisis de la red se pueden definir nodos internos. 
![Consigna1](circuitos_consigna.PNG)
1. Construir la [Matriz Admitancia Indefinida para cada caso](#MAI_T). En el [BS170 plantear el modelo lineal](#MAI_NMOS) visto en clase con admitancias y generadores de corriente, que tenga en cuenta: $Ciss, Coss, Crss y gm$ 
   
2. Para la red T se pide:

     a ) Obtener [$\frac{V_{23}}{V_{13}}$, $\frac{V_{21}}{V_{31}}$ y$\frac{V_{12}}{V_{32}}$](#Transferencias_V) explicitando el armado de los cofactores de segundo orden.
 
     b ) [Graficar y observar detenidamente la respuesta de módulo y fase para cada transferencia](#graficos_mod_fase_trasferencias). Analizar si entre [2 de las transferencias halladas se cumple que son complementarias en módulo :$|Ta+Tb|=1$](#T1+T2=1)
3. Para el MOS se pide:

     a ) Dada la MAI hallada en 1 obtener los [parámetros $Y$para las configuraciones source común y drain común.](#Parametros_Y_SC_DC) 

     b ) Calcular la [matriz de parámetros $ABCD$ en configuración source común](#Matriz_T_mos).

Propiedades de la MAI:   

La sumatoria de filas es igual a cero.

La sumatoria de columnas es igual a cero.

El determinante de la matriz es igual a cero.

Todos los cofactores de primer orden son iguales, es decir: $Y_j ^i=Y_m ^n $

Bonus

💎 Implementar en [Python el armado de la MAI](#MAI_Pyton) y la obtención de 2a) con los módulos disponibles en pytc2 

🥉 [Simulación Circuital y medición de los parámetros $Y$ del MOS en configuración drain común](#Simu_P_Y). Implementar el modelo SPICE del BS170:


## 1)Matriz MAI
#### Red Tee<a id="MAI_T"></a>

Para encontrar la matriz MAI se plantean las ecuaciones de los nodos, para facilitar el trabajo agregue un nodo N°4 en el medio de la Tee de modo que cada parametro de cada fila queda definido por solo un componente, simplificando su armado, por simple inspeccion se podia obtener la matriz una vez agregado este nodo y con tener en cuenta de que la suma de cada fila y columna debe de dar igual a 0
[![Matriz_MAI_T](Matriz_MAI_T.jpeg)

#### NMOS BS170<a id="MAI_NMOS"></a>
Para la matriz MAI del NMOS del BS170 se procedio de manera similar, sin embargo debido a la fuente de corriente el analisis no sale tan simple como en la red Tee por lo que las ecuaciones resultaron de mayor ayuda, una ventaja de este es que no hubo necesidad de agregar ningun nodo extra dentro de la red como en el caso anterior.
[![Matriz_MAI_NMOS](Matriz_MAI_NMOS.jpeg)


## 2)Red Tee
a) <a id="Trasnferencias_V"></a> Para la obtencion de las trasnferencias saque los cofactores de segundo orden segun la trasnferencia que pide cada caso. El primer cofactor se debe de elimiar las filas del dipolo de entrada de la transferencia y las columnas del dipolo de salida, mientras que para el cofactor que divide se deben eliminar filas y columnas del dipolo de entrada. Una vez realizado esto se debe de sacar el determinante de las matrices que quedan y realizar la division. Por ultimo como son trasnferencias se deben de multiplicar por el signo que se obtiene de la formula que se indica en cada caso.
![Trasnferencias V/V con Cofactores](Transferencias_V.jpeg)
![Trasnferencias V/V con Cofactores](Transferencias_V_2.jpeg)<a id="T1+T2=1"></a>

Para comparar a simple vista las transferencias puse de valores 1Hy y 1F en las bobinas y capacitores, excepto en la trasnferencia de V12/V32 que el cpaacitor vale 2F ya que sino seria exactamente igual a la trasnferencia de V21/31.
Se puede observar como estas dos trasnferencias mencionadas en las que ambas entran por el capacitor y salen por alguna de las bobinas se obtiene un pasa altos, mientras que si se entra por alguna  de las bobinas se obtiene un pasabajos como en la trasnferencia V23/V13

![Trasnferencias V/V con Cofactores](Transferencias.PNG)<a id="graficos_mod_fase_trasferencias"></a>

#### 3) NMOS
a) <a id="Parametros_Y_SC_DC"></a>Para la obtencion de las matrices admitancia, dado que ya estamos trabajando sobre una matriz de admitancias indefinida, basta con condicionar los puertos segun lo pedido, en el primer caso como pide un source comun se pone el Nodo N°3 a 0, esto se tradice en eliminar de la matriz la fila y columna N°3. Para la segunda en la que se pidio configuracion Drain comun, se debio de eliminar la fila y columna N°2. 
![Matriz Y configuracion soruce comun y drain comun de NMOS](Matriz_Y_SC_DC.jpeg)

b) <a id="Matriz_T_mos"></a>Para obtener la matriz T utilize la trasnformacion obtenida en la tarea semanal 5, donde se saco de las ecuaciones los parametros ABCD de los ya obtenidos en el punto anterior de la Y resultando de la siguiente manera:
![Matriz T configuracion soruce comun de NMOS](Matriz_T_NMOS.jpeg)


##### Codigo armado de la MAI<a id="MAI_Pyton"></a>

In [9]:
import sympy as sp
import numpy as np
from pytc2.sistemas_lineales import pretty_print_lti, pretty_print_SOS

# ---------------------------------------------------------------
# Símbolos
# ---------------------------------------------------------------
s, L1, L2, C = sp.symbols('s L1 L2 C')
# Nota: s representa jw. Para régimen senoidal permanente, sustituir
# s -> sp.I*w  (con w = sp.symbols('omega', positive=True))

# Admitancias de cada rama de la red
Y_L1 = 1 / (s * L1)   # bobina entre nodo 1 y nodo 4
Y_L2 = 1 / (s * L2)   # bobina entre nodo 4 y nodo 2
Y_C  = s * C          # capacitor entre nodo 4 y nodo 3


# ---------------------------------------------------------------
# 1) Construcción de la MAI (4 nodos: 1, 2, 3, 4 -> índices 0..3)
# ---------------------------------------------------------------
def add_admitancia(Y, i, j, y):
    """Suma la admitancia y entre los nodos i y j (índices 0-based) a la MAI."""
    Y[i, i] += y
    Y[j, j] += y
    Y[i, j] -= y
    Y[j, i] -= y


n = 4
Y = sp.zeros(n, n)

# índices: nodo1->0, nodo2->1, nodo3->2, nodo4->3
add_admitancia(Y, 0, 3, Y_L1)   # L1 entre 1 y 4
add_admitancia(Y, 3, 1, Y_L2)   # L2 entre 4 y 2
add_admitancia(Y, 3, 2, Y_C)    # C  entre 4 y 3

print("=== MAI completa (orden de nodos: 1, 2, 3, 4) ===")
sp.pprint(Y)

# chequeo: cada fila y cada columna deben sumar 0 (propiedad de toda MAI)
assert all(sp.simplify(sum(Y.row(i))) == 0 for i in range(n)), "fila no suma 0"
assert all(sp.simplify(sum(Y.col(j))) == 0 for j in range(n)), "columna no suma 0"


# ---------------------------------------------------------------
# 2) Transferencias de tensión en vacío (I_salida = 0)
#
#    Se "aterra" el nodo de referencia (el que se repite en numerador
#    y denominador), quedando una matriz 2x2 con los otros dos nodos.
#    Con I_salida = 0:  0 = Y[out,in]*V_in + Y[out,out]*V_out
#    =>  V_out/V_in = -Y[out,in] / Y[out,out]
# ---------------------------------------------------------------
def transferencia(Y_red, labels, referencia, entrada, salida):
    resto = [lbl for lbl in labels if lbl != referencia]
    pos_resto = [labels.index(lbl) for lbl in resto]
    Y2 = Y_red[pos_resto, pos_resto]          # matriz 2x2 con el nodo de referencia a masa

    idx = {lbl: resto.index(lbl) for lbl in resto}
    i_in, i_out = idx[entrada], idx[salida]

    return sp.simplify(-Y2[i_out, i_in] / Y2[i_out, i_out])


T_V23_V13 = transferencia(Y_red, new_labels, referencia=3, entrada=1, salida=2)
T_V21_V31 = transferencia(Y_red, new_labels, referencia=1, entrada=3, salida=2)
T_V12_V32 = transferencia(Y_red, new_labels, referencia=2, entrada=3, salida=1)

print("\n=== Transferencias ===\n")

V23_V13 = sp.Symbol('V23/V13')
V21_V31 = sp.Symbol('V21/V31')
V12_V32 = sp.Symbol('V12/V32')

sp.pretty_print(sp.Eq(V23_V13, T_V23_V13))
print()
sp.pretty_print(sp.Eq(V21_V31, T_V21_V31))
print()
sp.pretty_print(sp.Eq(V12_V32, T_V12_V32))


# ---------------------------------------------------------------
# 4) Impresión "bonita" con pytc2 (formato omega0 / Q)
#    pretty_print_SOS necesita coeficientes NUMÉRICOS (no símbolos),
#    por eso se reemplazan L1, L2, C por los valores de 'valores'.
#    Al ser una red LC pura (sin resistencias), Q -> infinito,
#    por eso el denominador queda expresado como (s^2 + w0^2).
# ---------------------------------------------------------------
#def sympy_a_sos(expr, s, valores):
#    """Convierte una función de transferencia simbólica de 2do orden
#    en una fila de SOS: [a1, a2, a3, b1, b2, b3] (coeficientes numéricos
#    de s^2, s^1, s^0 en numerador y denominador)."""
#   num, den = sp.fraction(sp.together(expr))

#   def coeficientes(poly_expr):
#       p = sp.Poly(poly_expr, s)
#        c = p.all_coeffs()
#       c = [0] * (3 - len(c)) + c            # completa a grado 2
#       return [float(sp.N(ci.subs(valores) if hasattr(ci, "subs") else ci)) for ci in c]

#   return coeficientes(num) + coeficientes(den)


#sos_V23_V13 = np.array([sympy_a_sos(T_V23_V13, s, valores)])
#sos_V21_V31 = np.array([sympy_a_sos(T_V21_V31, s, valores)])
#sos_V12_V32 = np.array([sympy_a_sos(T_V12_V32, s, valores)])

#print("\n=== Transferencias con pytc2 (formato omega0/Q) ===")
#print("(usar en Jupyter para ver el LaTeX renderizado)\n")

#print("V23/V13:")
#pretty_print_SOS(sos_V23_V13, mode='omegayq')

#print("V21/V31:")
#pretty_print_SOS(sos_V21_V31, mode='omegayq')

#print("V12/V32:")
#pretty_print_SOS(sos_V12_V32, mode='omegayq')

=== MAI completa (orden de nodos: 1, 2, 3, 4) ===
⎡ 1                      -1         ⎤
⎢────   0     0          ────       ⎥
⎢L₁⋅s                    L₁⋅s       ⎥
⎢                                   ⎥
⎢       1                -1         ⎥
⎢ 0    ────   0          ────       ⎥
⎢      L₂⋅s              L₂⋅s       ⎥
⎢                                   ⎥
⎢ 0     0    C⋅s         -C⋅s       ⎥
⎢                                   ⎥
⎢-1    -1                 1      1  ⎥
⎢────  ────  -C⋅s  C⋅s + ──── + ────⎥
⎣L₁⋅s  L₂⋅s              L₂⋅s   L₁⋅s⎦

=== Transferencias ===

               1     
V23/V13 = ───────────
                2    
          C⋅L₁⋅s  + 1

                  2  
            C⋅L₁⋅s   
V21/V31 = ───────────
                2    
          C⋅L₁⋅s  + 1

                  2  
            C⋅L₂⋅s   
V12/V32 = ───────────
                2    
          C⋅L₂⋅s  + 1


 #### Simulacion del transistor y sus parametros <a id="Simu_P_Y"></a>
 Aqui se simulo la entrada y salida de un transistor con el modelo presentado por el profesor, se pueden ver cosas interesentes como el hecho de que ya no son constantes estos parametros sino que pareciera que divergen, obviamente el limite los pone luego la fisica del dispositivo, por otro lado mencionar que el Y21 es igual al Y12 por lo que este ultimo no se ve al quedar tapado por el primero. 
 ![Simulacion de transistor en drain comun y obtencion de parametros Y](Y_Param_DrainC_MOS.PNG)
